# 직접 해보기 06 · 이산화탄소의 봉우리 재기

WASP-39 b의 투과 스펙트럼(JWST ERS 팀, Zenodo 10.5281/zenodo.6959427, CC BY 4.0)을 읽는다. **한 번의 관측**을 네 팀이 각자 처리한 결과다.

같은 계산이 `analysis.py`에도 들어 있다.

In [ ]:
import analysis as A
rows = A.load()
print(A.sha8(), len(rows), '행')
for p in A.PIPES:
    print(p, sum(1 for r in rows if r['pipe'] == p and r['flag'] == 0), '점 (표시 점 제외)')

## 1. 네 처리 결과를 같은 눈금으로 그린다
matplotlib이 없으면 `pip install matplotlib`. 음영이 두 구간이다.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 1, figsize=(8, 9), sharex=True)
for ax, p in zip(axes, A.PIPES):
    pts = [r for r in rows if r['pipe'] == p and r['flag'] == 0]
    ax.plot([r['w'] for r in pts], [r['d'] for r in pts], '.', c='k', ms=3)
    ax.axvspan(3.6, 3.9, color='C0', alpha=.15); ax.axvspan(4.2, 4.5, color='C0', alpha=.15)
    ax.set_ylim(20400, 23200); ax.set_ylabel(p)
axes[-1].set_xlabel('파장 (μm)'); plt.tight_layout()

## 2. 두 구간 평균의 차이 (책의 예상 결과)
구간은 [시작, 끝)이다. 시작은 넣고 끝은 뺀다.

In [ ]:
for p in A.PIPES:
    r = A.summarize(rows, p)
    print(f"{p:<9} 가장 깊은 점 {r['deepest_um']:.3f}μm | {r['n_feature']}점 / {r['n_compare']}점 | 차이 {r['diff']:,.0f} ppm")

## 3. 경계와 처리팀을 바꿔 본다
특징 구간을 4.3–4.4로 좁히거나 비교 구간을 5.0–5.3으로 옮기면? 원하는 답이 나올 때까지 옮겼다면 사후 탐색이라고 적어야 한다.

In [ ]:
for feat, comp in [((4.2, 4.5), (3.6, 3.9)), ((4.3, 4.4), (3.6, 3.9)), ((4.2, 4.5), (5.0, 5.3))]:
    print(feat, comp, [round(A.summarize(rows, p, feat, comp)['diff']) for p in A.PIPES])

## 4. TIBERIUS의 표시 점
불량 픽셀로 표시된 18점을 넣으면 결과가 어떻게 달라지는가?

In [ ]:
print(round(A.summarize(rows, 'TIBERIUS')['diff']), round(A.summarize(rows, 'TIBERIUS', keep_flagged=True)['diff']))